In [ ]:
import os
# ===================== 配置中国镜像 =====================
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

import sys
import torch
from PIL import Image
from transformers import (
    Blip2Processor,
    Blip2ForConditionalGeneration
)

# 1. 获取项目根目录（当前目录的父目录）
project_root = os.path.dirname(os.getcwd())
# 2. 将根目录加入系统路径
sys.path.append(project_root)
os.chdir(project_root)


class BLIP2Runner:
    def __init__(
        self, 
        model_name="Salesforce/blip2-flan-t5-xl",
        cache_dir="./blip2_cache", 
        device=None,
        dtype=torch.float16
    ):
        self.model_name = model_name
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)

        # 自动选择设备
        if device:
            self.device = torch.device(device)
        else:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # processor
        self.processor = Blip2Processor.from_pretrained(
            model_name,
            cache_dir=cache_dir
        )

        # model
        self.model = Blip2ForConditionalGeneration.from_pretrained(
            model_name,
            cache_dir=cache_dir,
            torch_dtype=dtype,
            device_map="auto" if self.device.type == "cuda" else None
        )

        self.model.eval()

    def load_image(self, image_path):
        """加载图片"""
        return Image.open(image_path).convert("RGB")

    @torch.no_grad()
    def generate_caption(self, image, prompt=None, max_length=50):
        """
        生成图片描述
        Args:
            image: 图片路径或 PIL.Image 对象
            prompt: 引导生成的提示词
        """
        if isinstance(image, str):
            image = self.load_image(image)

        # 构建输入
        if prompt:
            inputs = self.processor(images=image, text=prompt, return_tensors="pt").to(self.device)
        else:
            inputs = self.processor(images=image, return_tensors="pt").to(self.device)

        # 生成描述
        generated_ids = self.model.generate(
            **inputs,
            max_length=max_length,
            num_beams=5,
            do_sample=False,
            early_stopping=True
        )
        
        generated_text = self.processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        return generated_text




/home/feng1/miniconda3/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===================== 测试示例 =====================
# 初始化模型
blip2_runner = BLIP2Runner(cache_dir="./model/blip/blip2_cache")
test_image = "./model/test/test/1000268201.jpg"  # 请确保此图片路径存在

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 1289/1289 [00:32<00:00, 40.05it/s, Materializing param=vision_model.post_layernorm.weight]                                                
The tied weights mapping and config for this model specifies to tie language_model.shared.weight to language_model.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [3]:
print(f"正在测试图片: {test_image}\n")

# 定义不同的提示词进行测试
prompts = [
        "",                      # 无提示词（默认生成）
        "A photo of",            # 这是一个...
        "A realistic photo of",  # 一张写实的...
        "An image showing"       # 一张展示...
]

print("=== 提示词生成测试 ===")
for prompt in prompts:
    caption = blip2_runner.generate_caption(test_image, prompt=prompt)
    print(f"提示词: '{prompt}'")
    print(f"生成结果: {caption}\n")

正在测试图片: ./model/test/test/1000268201.jpg

=== 提示词生成测试 ===
提示词: ''
生成结果: a little girl standing on the steps of a chicken coop

提示词: 'A photo of'
生成结果: a little girl standing in front of a chicken coop

提示词: 'A realistic photo of'
生成结果: a little girl standing on the steps of a chicken coop

提示词: 'An image showing'
生成结果: a little girl standing in front of a chicken coop

